In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from ilipy import LogLevel, PipeDistance, PipeDistanceRange, Session

from ilipyutils.tubeviews import MultiTrackTubeview, dataarray_to_image

# Setup

In [ ]:
session = Session("prod")
session.settings.set_log_level(LogLevel.off)
session.set_active_inspection("09QWB8A52AN")

# Track Selection

In [ ]:
# tracks = {0, 1, 2}
tracks = None  # all tracks

# Tubeviews

In [ ]:
mttv = MultiTrackTubeview(
    session=session,
    tracks=tracks,
    tubeview_engine="TubeView",
    tubeview_engine_kwargs={"masking": "OD"},
)

# Distances

In [ ]:
# Start and end
PD_START_M = 7257.0
PD_END_M = 7264.0
DISTANCE_RANGE = PipeDistanceRange(PipeDistance(PD_START_M), PipeDistance(PD_END_M))

# Step length
PD_STEP_LENGTH = None  # PipeDistance(1.0)

# Step width
n_frames_step = 500  # Approximate number of frames per batch
PD_STEP_PADDING = None  # PipeDistance(n_frames_step * 1.5e-3)
# assert PD_STEP_PADDING < PD_STEP_LENGTH, (
#     "If step width >= step length, it's better to do sequential"
# )

# Process

After this process, we get a dict of DataArrays containing all frames which were sampled.

In [ ]:
tubeviews_da = mttv.process(
    distance_range=DISTANCE_RANGE,
    step_length=PD_STEP_LENGTH,
    step_padding=PD_STEP_PADDING,
    show_progress=True,
    max_workers=4,
)

# Rasterize

We need to employ a gridding to ensure that all tracks are sampled at the same pipe_distance locations. Before we do that, we will use all possible data (the ungridded) to equalize channels.

In [ ]:
tubeview_cheq_da = mttv.equalize_lateral(tubeviews_da)

In [ ]:
resolution = PD_STEP_LENGTH or PipeDistance(1.5e-3)
tubeviews_reg_da = mttv.rasterize(
    tubeview_cheq_da,
    distance_range=DISTANCE_RANGE,
    step_length=resolution,
    max_gap_size=None,  # Ignore gaps
    agg="mean",
)

# Improving per-track amplitude balacing

We can apply a simple per-track bulk amplitude scaling, or equalize histogram. We need to make sure we use a clean section of pipe to do these operations.

In [ ]:
ref_region = slice(
    round(len(next(iter(tubeviews_reg_da.values()))) * 0.9),
    None,
)

In [ ]:
# Per-track normalization based on a reference region if we want
# This step is not necessary when using histogram equalization
tubeviews_reg_da = {
    track: tubeview.fillna(0) / np.nanmedian(tubeview[ref_region, :])
    for track, tubeview in tubeviews_reg_da.items()
}

In [ ]:
# Histogram equalization
tubeviews_reg_hist_eq_da = mttv.equalize_histogram(
    tubeviews_reg_da,
    references={
        track: (da := tubeview[ref_region, :].to_numpy()[:])[np.isfinite(da)]
        for track, tubeview in tubeviews_reg_da.items()
    },
)

In [ ]:
# Create a PIL
tubeviews_reg_hist_eq_PIL = {
    track: dataarray_to_image(
        tubeview.sortby("tool_lateral_deg").T, how="linear"
    ).to_pil(origin="upper")
    for track, tubeview in tubeviews_reg_hist_eq_da.items()
}

In [ ]:
for track, pil_img in tubeviews_reg_hist_eq_PIL.items():
    pil_img.save(
        f"tubeview_clock_{mttv._tool_angle_dict[track]:.0f}deg_track_{track}.png"
    )

# Combine

In [ ]:
full_tubeview_reg_hist_eq_da = mttv.combine(
    tubeviews_reg_hist_eq_da,
    mapping="interpolate",
    y_coord="tool_lateral_deg",
    rasterize_agg="cubic",
)

In [ ]:
# Create a PIL
full_tubeview_reg_hist_eq_PIL = dataarray_to_image(
    full_tubeview_reg_hist_eq_da.sortby("tool_lateral_deg").T, how="eq_hist"
).to_pil(origin="upper")
full_tubeview_reg_hist_eq_PIL.save("tubeview_full_hist_eq.png")

In [ ]:
full_tubeview_reg_hist_eq_PIL

In [ ]:
da = full_tubeview_reg_hist_eq_da
x = "pipe_distances_m"
y = "tool_lateral_deg"
pclip = 0.5
fig, ax = plt.subplots(figsize=(14, 14 * da.shape[1] / da.shape[0]))
ax.imshow(
    da.T,
    cmap="gray",
    aspect="auto",
    extent=(da[x][0].item(), da[x][-1].item(), da[y][-1].item(), da[y][0].item()),
    vmin=0,
    vmax=da.max().item() * pclip,
    interpolation="nearest",
)